In [45]:
# Extract Fedorenko fROIs - Per Session
"""
Extract top 10% voxels from Fedorenko language parcels
Works with unified GLM results structure
Creates session-specific fROI masks
"""
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import nibabel as nib
from nilearn.image import resample_to_img
from nilearn import plotting
import matplotlib.pyplot as plt

# =============================================================================
# CONFIGURATION
# =============================================================================

ROOT = Path("/scratch/ibilgin/cneuromod.langlocalizer.fmriprep")
GLM_ROOT = ROOT / "langlocalizer_glm_results"

SUBJECTS = ["sub-01", "sub-02", "sub-03", "sub-05"]

TASK_CONTRASTS = {
    "aliceFr": ["int-degr"],
    "aliceEn": ["int-degr"],
    "listening": ["int-degr"],
    "reading": ["word-nonword"], 

# Load Fedorenko atlas
FED_ATLAS_NII = Path("/scratch/ibilgin/fedorenko_atlas/allParcels-language-SN220.nii")
FED_ATLAS_TXT = Path("/scratch/ibilgin/fedorenko_atlas/allParcels-language-SN220.txt")

TOP_PERCENT = 0.20
TOP_VOXELS = 20

OUT_DIR = ROOT / f"fedorenko_fROIs_top{TOP_VOXELS}_updated"
OUT_DIR.mkdir(exist_ok=True, parents=True)

PNG_DIR = OUT_DIR / "qc_png_updated"
PNG_DIR.mkdir(exist_ok=True, parents=True)

LANG_GLM = ROOT / "langlocalizer_glm_results"
FIG_DIR = ROOT / f"figures_integrated_analysis_{TOP_VOXELS}_top"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("="*80)
print("FEDORENKO fROI EXTRACTION")
print("="*80)
print(f"GLM results: {GLM_ROOT}")
print(f"Output: {OUT_DIR}")
print(f"Subjects: {SUBJECTS}")
print(f"Top {int(TOP_PERCENT*100)}% voxels per parcel")


allLANG_MASK_DIR = OUT_DIR / f"allLANG_masks_session_top_{TOP_VOXELS}"
allLANG_FIG_DIR = OUT_DIR / f"allLANG_figures_session_top_{TOP_VOXELS}"
allLANG_MASK_DIR.mkdir(exist_ok=True, parents=True)
allLANG_FIG_DIR.mkdir(exist_ok=True, parents=True)


TASK_COLORMAPS = {
    "aliceFr": "Greens",
    "aliceEn": "Blues",
    "listening": "Purples",
    "reading": "Reds",
}



FEDORENKO fROI EXTRACTION
GLM results: /scratch/ibilgin/cneuromod.langlocalizer.fmriprep/langlocalizer_glm_results
Output: /scratch/ibilgin/cneuromod.langlocalizer.fmriprep/fedorenko_fROIs_top10_updated
Subjects: ['sub-01', 'sub-02', 'sub-03', 'sub-05']
Top 10% voxels per parcel


In [43]:

# =============================================================================
# LOAD FEDORENKO LABELS
# =============================================================================

def load_fed_labels(txt_path):
    """Load Fedorenko parcel names from text file."""
    if not txt_path.exists():
        raise FileNotFoundError(f"Atlas labels not found: {txt_path}")
    
    with open(txt_path, "r") as f:
        lines = [ln.strip() for ln in f.readlines() if ln.strip()]
    
    # 1-based indexing
    label_to_name = {i: name for i, name in enumerate(lines, start=1)}
    
    print(f"\nLoaded {len(label_to_name)} Fedorenko parcels:")
    for label, name in label_to_name.items():
        print(f"  {label:2d} -> {name}")
    
    return label_to_name

FED_LABELS = load_fed_labels(FED_ATLAS_TXT)

# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def load_fed_atlas(resample_target_img):
    """Load and resample Fedorenko atlas to match target image."""
    if not FED_ATLAS_NII.exists():
        raise FileNotFoundError(f"Atlas not found: {FED_ATLAS_NII}")
    
    atlas_img = nib.load(str(FED_ATLAS_NII))
    
    # Resample if needed
    if (atlas_img.shape != resample_target_img.shape or 
        not np.allclose(atlas_img.affine, resample_target_img.affine)):
        print("  [Resampling atlas to match contrast space...]")
        atlas_img = resample_to_img(
            atlas_img, resample_target_img, interpolation="nearest"
        )
    
    atlas_data = atlas_img.get_fdata()
    return atlas_data, atlas_img


def make_top_voxel_mask(contrast_data, parcel_mask, positive_only=True, top_p=TOP_PERCENT):
    """Select top p% voxels within a parcel."""
    vals = contrast_data[parcel_mask]
    
    if positive_only:
        vals = vals[vals > 0]
    
    if vals.size == 0:
        return np.zeros_like(contrast_data, dtype=bool)
    
    # Top p% threshold
    k = max(1, int(np.floor(top_p * vals.size)))
    sorted_vals = np.sort(vals)
    thresh = sorted_vals[-k]
    
    top_mask = parcel_mask & (contrast_data >= thresh)
    return top_mask
    
def save_qc_glassbrain(mask_img, subject, session, task, contrast, roi_name):
    """Save QC figure for each fROI mask."""
    fname = f"{subject}_{session}_task-{task}_contrast-{contrast}_parcel-{roi_name}.png"
    out_png = PNG_DIR / fname
    
    display = plotting.plot_glass_brain(
        mask_img,
        cmap="Greens",
        bg_img=None,
        black_bg=False,
        plot_abs=False,
        colorbar=False,
        display_mode="lyrz",
        title=f"{subject} {session} {task} {contrast} {roi_name}"
    )
    display.savefig(str(out_png), dpi=150, bbox_inches="tight")
    display.close()  


# =============================================================================
# SESSION-LEVEL PROCESSING
# =============================================================================

def process_session_frois(subject, session, task, contrast):
    """
    Extract fROIs for one session.
    
    Uses session-level z-maps from unified GLM structure:
    GLM_ROOT/subject/session/task/subject_session_task-task_contrast-contrast_stat-z.nii.gz
    """
    # Find z-map
    task_dir = GLM_ROOT / subject / session / task
    z_map_path = task_dir / f"{subject}_{session}_task-{task}_contrast-{contrast}_stat-z.nii.gz"
    
    if not z_map_path.exists():
        print(f"    [SKIP] Z-map not found: {z_map_path.name}")
        return
    
    print(f"    Processing: {z_map_path.name}")
    
    # Load z-map
    img = nib.load(str(z_map_path))
    data = img.get_fdata()
    
    # Load atlas
    atlas_data, _ = load_fed_atlas(img)
    
    # Process each parcel
    for label, roi_name in FED_LABELS.items():
        parcel_mask = (atlas_data == float(label)).astype(bool)
        n_vox = int(parcel_mask.sum())
        
        if n_vox == 0:
            continue
        
        # Get top 10%
        top_mask = make_top_voxel_mask(data, parcel_mask, positive_only=True, top_p=TOP_PERCENT)
        n_top = int(top_mask.sum())
        
        if n_top == 0:
            continue
        
        print(f"      {roi_name}: {n_vox} voxels → top {n_top}")
        
        # Save mask
        out_fname = (
            f"{subject}_{session}_task-{task}_contrast-{contrast}_"
            f"parcel-{roi_name}_top{TOP_VOXELS}.nii.gz"
        )
        out_path = OUT_DIR / out_fname
        
        mask_img = nib.Nifti1Image(top_mask.astype(np.uint8), img.affine, img.header)
        nib.save(mask_img, str(out_path))
        
        # QC figure
        save_qc_glassbrain(mask_img, subject, session, task, contrast, roi_name)

# =============================================================================
# SUBJECT-LEVEL (ACROSS SESSIONS) PROCESSING
# =============================================================================

def compute_mean_contrast(subject, task, contrast):
    """
    Compute mean z-map across all sessions for a subject/task/contrast.
    Uses subject-level GLM results if available, otherwise averages sessions.
    """
    # Try subject-level GLM first
    subj_level_dir = GLM_ROOT / subject / "subject_level" / task
    subj_level_map = subj_level_dir / f"{subject}_task-{task}_contrast-{contrast}_stat-z.nii.gz"
    
    if subj_level_map.exists():
        print(f"  [Using subject-level GLM]: {subj_level_map.name}")
        img = nib.load(str(subj_level_map))
        return img.get_fdata(), img
    
    # Otherwise, average session-level maps
    print(f"  [Averaging sessions]")
    subj_root = GLM_ROOT / subject
    
    vols = []
    ref_img = None
    
    for ses_dir in sorted(subj_root.glob("ses-*")):
        task_dir = ses_dir / task
        z_map = task_dir / f"{subject}_{ses_dir.name}_task-{task}_contrast-{contrast}_stat-z.nii.gz"
        
        if z_map.exists():
            print(f"    Including: {z_map.name}")
            img = nib.load(str(z_map))
            vols.append(img.get_fdata())
            ref_img = img
    
    if not vols:
        raise FileNotFoundError(f"No z-maps found for {subject}/{task}/{contrast}")
    
    mean_vol = np.mean(np.stack(vols, axis=0), axis=0)
    return mean_vol, ref_img


def process_subject_frois(subject, task, contrast):
    """Extract fROIs using subject-level (across-session) z-maps."""
    print(f"\n  Subject-level fROIs: {subject}/{task}/{contrast}")
    
    try:
        mean_data, ref_img = compute_mean_contrast(subject, task, contrast)
    except FileNotFoundError as e:
        print(f"    [SKIP] {e}")
        return
    
    # Load atlas
    atlas_data, _ = load_fed_atlas(ref_img)
    
    # Process each parcel
    for label, roi_name in FED_LABELS.items():
        parcel_mask = (atlas_data == float(label)).astype(bool)
        n_vox = int(parcel_mask.sum())
        
        if n_vox == 0:
            continue
        
        top_mask = make_top_voxel_mask(mean_data, parcel_mask, positive_only=True, top_p=TOP_PERCENT)
        n_top = int(top_mask.sum())
        
        if n_top == 0:
            continue
        
        print(f"    {roi_name}: {n_vox} voxels → top {n_top}")
        
        # Save mask
        out_fname = (
            f"{subject}_task-{task}_contrast-{contrast}_"
            f"parcel-{roi_name}_top{TOP_VOXELS}_MEAN.nii.gz"
        )
        out_path = OUT_DIR / out_fname
        
        mask_img = nib.Nifti1Image(top_mask.astype(np.uint8), ref_img.affine, ref_img.header)
        nib.save(mask_img, str(out_path))

# =============================================================================
# MAIN LOOP
# =============================================================================

print("\n" + "="*80)
print("PROCESSING SESSION-LEVEL fROIs")
print("="*80)

for subject in SUBJECTS:
    print(f"\n{'='*60}\n{subject}\n{'='*60}")
    
    # Find all sessions for this subject
    subj_root = GLM_ROOT / subject
    
    for task, contrast_list in TASK_CONTRASTS.items():
        for contrast in contrast_list:
            print(f"\n  Task: {task}, Contrast: {contrast}")
            
            # Process each session
            for ses_dir in sorted(subj_root.glob("ses-*")):
                session = ses_dir.name
                print(f"  Session: {session}")
                
                process_session_frois(subject, session, task, contrast)

print("\n" + "="*80)
print("PROCESSING SUBJECT-LEVEL fROIs (ACROSS SESSIONS)")
print("="*80)

for subject in SUBJECTS:
    print(f"\n{'='*60}\n{subject}\n{'='*60}")
    
    for task, contrast_list in TASK_CONTRASTS.items():
        for contrast in contrast_list:
            process_subject_frois(subject, task, contrast)

print("\n" + "="*80)
print("COMPLETE")
print("="*80)
print(f"fROI masks saved to: {OUT_DIR}")
print(f"QC figures saved to: {PNG_DIR}")

# Count outputs
session_masks = list(OUT_DIR.glob(f"*ses-*_parcel-*_top{TOP_VOXELS}.nii.gz"))
subject_masks = list(OUT_DIR.glob("*_MEAN.nii.gz"))
print(f"\nCreated:")
print(f"  - {len(session_masks)} session-level fROI masks")
print(f"  - {len(subject_masks)} subject-level fROI masks")


Loaded 10 Fedorenko parcels:
   1 -> LH_IFGorb
   2 -> LH_IFG
   3 -> LH_MFG
   4 -> LH_AntTemp
   5 -> LH_PostTemp
   6 -> RH_IFGorb
   7 -> RH_IFG
   8 -> RH_MFG
   9 -> RH_AntTemp
  10 -> RH_PostTemp

PROCESSING SESSION-LEVEL fROIs

sub-01

  Task: aliceFr, Contrast: int-degr
  Session: ses-001
    Processing: sub-01_ses-001_task-aliceFr_contrast-int-degr_stat-z.nii.gz
  [Resampling atlas to match contrast space...]
      LH_IFGorb: 370 voxels → top 47
      LH_IFG: 743 voxels → top 121
      LH_MFG: 462 voxels → top 40
      LH_AntTemp: 1627 voxels → top 210
      LH_PostTemp: 2948 voxels → top 365
      RH_IFGorb: 644 voxels → top 70
      RH_IFG: 370 voxels → top 49
      RH_MFG: 743 voxels → top 103
      RH_AntTemp: 462 voxels → top 29
      RH_PostTemp: 1627 voxels → top 174
  Session: ses-002
    Processing: sub-01_ses-002_task-aliceFr_contrast-int-degr_stat-z.nii.gz
  [Resampling atlas to match contrast space...]
      LH_IFGorb: 370 voxels → top 28
      LH_IFG: 743 voxels

In [6]:
import json
import os

# Check for participants.tsv anywhere in the dataset
base = "/scratch/ibilgin/cneuromod.langlocalizer.fmriprep/sourcedata/cneuromod.langlocalizer/"

for root, dirs, files in os.walk(base):
    for f in files:
        if "participants" in f or "phenotype" in f.lower():
            fpath = os.path.join(root, f)
            print(fpath)
            with open(fpath) as fp:
                print(fp.read())
    # Only check top 2 levels
    if root.count(os.sep) - base.count(os.sep) >= 2:
        break

/scratch/ibilgin/cneuromod.langlocalizer.fmriprep/sourcedata/cneuromod.langlocalizer/participants.json
{
  "participant_id": {
    "Description": "Participant identifier"
  },
  "age": {
    "Description": "Age in years (TODO - verify) as in the initial session, might not be correct for other sessions"
  },
  "sex": {
    "Description": "self-rated by participant, M for male/F for female (TODO: verify)"
  },
  "group": {
    "Description": "(TODO: adjust - by default everyone is in control group)"
  }
}
/scratch/ibilgin/cneuromod.langlocalizer.fmriprep/sourcedata/cneuromod.langlocalizer/participants.tsv
participant_id	age	sex	group
sub-01	47	M	control

